# basic
https://github.com/jiminelee/agent-harness-course/blob/main/module03/module03_tool_registry.py

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
from openai import OpenAI
import json

BASE_URL = "http://localhost:11434/v1"
API_KEY = "ollama"
MODEL = "qwen2.5:7b"
LLM_OPTIONS = {"reasoning_effort": "none"} if API_KEY == "ollama" else {}
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)


# system prompt saying there are many tools

In [3]:
MAX_TURNS = 8

SYSTEM_PROMPT = """You are a helpful assistant with access to multiple tools:
calculate, search_web, and read_file. Choose the right tool for each part of
the task. Only give a final answer once you have everything you need.
"""

# Tool registery pattern, can use tool decorator to register tools in bulk


In [4]:
TOOL_REGISTRY = {}   # name -> python function
TOOLS_SCHEMA = []    # name -> JSON schema description sent to the LLM


def tool(description: str, parameters: dict, required: list):
    """
    Decorator factory that registers a function as an agent tool.

    - description: human/LLM-readable explanation of what the tool does.
    - parameters: JSON-schema "properties" dict describing each argument.
    - required: list of argument names that must be provided.
    """
    def decorator(fn):
        TOOL_REGISTRY[fn.__name__] = fn
        TOOLS_SCHEMA.append({
            "type": "function",
            "function": {
                "name": fn.__name__,
                "description": description,
                "parameters": {
                    "type": "object",
                    "properties": parameters,
                    "required": required,
                },
            },
        })
        return fn
    return decorator



# register the tools with this decorator: the function def must follow the decorator 

In [15]:
@tool(
    description="Evaluate a basic arithmetic expression and return the numeric result.",
    parameters={
        "expression": {"type": "string", "description": "e.g. '2 + 2' or '(5*3)/2'"}
    },
    required=["expression"],
)
def calculate(expression: str) -> str:
    """Same as Module 2's calculator tool."""
    result = eval(expression, {"__builtins__": {}}, {})
    return str(result)


@tool(
    description="Search the web for a query and return a short summary of results.",
    parameters={
        "query": {"type": "string", "description": "The search query string."}
    },
    required=["query"],
)
def search_web(query: str) -> str:
    """
    STUB IMPLEMENTATION for learning purposes only.

    In a real harness, this function would call a real search API
    (e.g. Tavily, Bing Search API, SerpAPI) and return actual results.
    Here we just return a fake canned string so you can see the tool-calling
    mechanics work end-to-end without needing an API key for a search
    provider on top of your LLM provider.
    """
    return f"No connection to the internet, the web search can not be executed."


@tool(
    description="Read the full contents of a local text file, given its path.",
    parameters={
        "path": {"type": "string", "description": "Filesystem path to a text file."}
    },
    required=["path"],
)
def read_file(path: str) -> str:
    """
    A REAL tool (unlike search_web above) -- actually reads from disk.
    Demonstrates proper error handling: a missing file should NOT crash
    the whole agent loop, it should return an error string the LLM can react to.
    """
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

# tool call and error handling
Executes a single tool call safely, covering all three failure modes
described at the top of this file. Always returns a string (never
raises), because a tool result must be feedable back to the LLM no
matter what happened.

In [16]:
def execute_tool_call(tool_call) -> str:
    # 0. take the tool call object name
    tool_name = tool_call.function.name
    
    # 1. check if the tool is in the registry
    tool_fn = TOOL_REGISTRY.get(tool_name)
    if tool_fn is None:
        return f"ERROR: unknown tool '{tool_name}'. Available tools: {list(TOOL_REGISTRY.keys())}"

    # Failure mode 2: the arguments aren't valid JSON, or are missing
    # required fields. We check this BEFORE calling the function so a bad
    # call doesn't produce a confusing Python-level TypeError.
    try:
        tool_args = json.loads(tool_call.function.arguments)
    except json.JSONDecodeError as e: # arguments aren't valid JSON
        return f"ERROR: could not parse arguments as JSON: {e}" 

    try:
        return str(tool_fn(**tool_args)) # execute the tool if it can

    # arguments are missing required fields
    except TypeError as e: 
        # Usually means required args were missing or extra/unknown args
        # were passed -- a mismatch between what the model sent and what
        # the function signature expects.
        return f"ERROR: invalid arguments for '{tool_name}': {e}"

    except Exception as e: # tool internal error
        # Failure mode 3: the tool ran but threw an exception internally
        # (e.g. read_file on a path that doesn't exist).
        # We deliberately don't crash the agent loop -- we hand the error
        # back to the model as a string so it can decide what to do next
        # (e.g. try a different path, apologize to the user, etc.).
        return f"ERROR: tool '{tool_name}' raised an exception: {e}"


# agent loop with tool error handling and tools calling

In [21]:
def run_agent_loop(user_task: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_task},
    ]

    for turn in range(1, MAX_TURNS + 1):
        print(f"\n{'=' * 60}")
        print(f"[STEP {turn}] Sending {len(messages)} messages to the LLM (tools enabled)...")
        print(f"{'=' * 60}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS_SCHEMA,
            temperature=0.0,
            **LLM_OPTIONS,
        )

        message = response.choices[0].message
    
        # Case 1: the model wants to call one or more tools.    
        if message.tool_calls:
            print(f"[STEP {turn}] LLM requested {len(message.tool_calls)} tool call(s).")
            messages.append(message)
            
            for tool_call in message.tool_calls:
                print(f"  -> Tool requested: '{tool_call.function.name}' "
                      f"args={tool_call.function.arguments}")

                
                # All error handling now lives inside execute_tool_call,
                # keeping this loop body simple and readable.
                tool_result = execute_tool_call(tool_call)
                print(f"  <- Result: {tool_result[:200]}"
                      f"{'...' if len(tool_result) > 200 else ''}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result,
                })
            

        # Case 2: the model gave a normal text answer -- no tool call means
        # it considers itself done (this is the standard convention for
        # tool-calling APIs: absence of tool_calls == final answer).
        else:
            print(f"\n[STEP {turn}] Model gave a final answer (no tool calls).")
            messages.append(message)
            return message.content
        
    print("\n[WARNING] Max turns reached without a final answer.")
    return "(Agent did not finish within the turn limit.)"
    

# run

In [22]:
task = (
        "Search the web for 'current agent harness best practices', then "
        "calculate 15% of 240, and combine both findings into one short summary."
    )
print(f"USER TASK: {task}")

result = run_agent_loop(task)

print(f"\n{'#' * 60}")
print("FINAL ANSWER:")
print(result)
print(f"{'#' * 60}")

USER TASK: Search the web for 'current agent harness best practices', then calculate 15% of 240, and combine both findings into one short summary.

[STEP 1] Sending 2 messages to the LLM (tools enabled)...
[STEP 1] LLM requested 2 tool call(s).
  -> Tool requested: 'search_web' args={"query":"current agent harness best practices"}
  <- Result: No connection to the internet, the web search can not be executed.
  -> Tool requested: 'calculate' args={"expression":"240 * 0.15"}
  <- Result: 36.0

[STEP 2] Sending 5 messages to the LLM (tools enabled)...

[STEP 2] Model gave a final answer (no tool calls).

############################################################
FINAL ANSWER:
Since the web search couldn't be executed due to lack of internet connection, I will provide a summary based on the calculation result. 

15% of 240 is 36.0. Without the web search results, I cannot include current best practices for agent harnesses in the summary. If you have any other questions or need further a